# Value iteration on the 3x3 grid world

Policy iteration alternated two phases: evaluate $\pi$ until $\Delta < \theta$, then improve.
Value iteration collapses them into **one** update by putting the $\max$ inside the sweep,

$$V(s) \leftarrow \max_a \sum_{s', r} p(s', r \mid s, a)\left[ r + \gamma V(s') \right],$$

which is just the Bellman *optimality* equation turned into an assignment. There is no
policy to carry around: the policy is read off once, at the very end.

Note on the pseudocode box: it says to initialize $V$ arbitrarily "except that
$V(\text{terminal}) = 0$". Our task is **continuing** ($s_9$ is not absorbing) so there is no terminal state and that clause simply does not apply.

In [1]:
import numpy as np

## The environment

|        | col 1 | col 2 | col 3 |
|--------|-------|-------|-------|
| row 1  | $s_1$ white | $s_2$ white | $s_3$ white |
| row 2  | $s_4$ white | $s_5$ white | $s_6$ **orange** |
| row 3  | $s_7$ **orange** | $s_8$ white | $s_9$ **blue (target)** |

* 5 actions: `0 = up`, `1 = right`, `2 = down`, `3 = left`, `4 = stay`.
* Transitions are deterministic.
* The reward depends on the cell you *enter*: $+1$ for the blue cell, $-1$ for an orange
  cell, $0$ for a white cell, and $-1$ if the action would take you off the grid
  (in that case you also stay where you are).
* Continuing task: $s_9$ is **not** absorbing, staying there keeps paying $+1$.

In [2]:
# Parameters
gamma         = 0.9      # discount factor
theta         = 1e-6     # accuracy threshold

n_states      = 9
n_actions     = 5

action_names  = ["up", "right", "down", "left", "stay"]
action_arrows = ["^", ">", "v", "<", "o"]

## The model

Since everything is deterministic, the model
$p(s', r | s, a)$ collapses into two $9 \times 5$ tables:

$$\texttt{NEXT}[s, a] = s' \qquad \texttt{REWARD}[s, a] = r.$$

Rows are states $s_1,\ldots,s_9$, columns are the five actions. Python indexes
from 0, so state $s_i$ lives at row `i-1` (e.g. $s_9$ is row `8`).

In [3]:
# Next state s' reached from state s (row) by action a (column).
NEXT = np.array([
#   up  right  down  left  stay
    [0,   1,    3,    0,    0],   # s1
    [1,   2,    4,    0,    1],   # s2
    [2,   2,    5,    1,    2],   # s3
    [0,   4,    6,    3,    3],   # s4
    [1,   5,    7,    3,    4],   # s5
    [2,   5,    8,    4,    5],   # s6  (orange)
    [3,   7,    6,    6,    6],   # s7  (orange)
    [4,   8,    7,    6,    7],   # s8
    [5,   8,    8,    7,    8],   # s9  (blue target)
])

# Reward collected on that transition.
REWARD = np.array([
#   up  right  down  left  stay
    [-1,   0,    0,   -1,    0],  # s1
    [-1,   0,    0,    0,    0],  # s2
    [-1,  -1,   -1,    0,    0],  # s3
    [ 0,   0,   -1,   -1,    0],  # s4
    [ 0,  -1,    0,    0,    0],  # s5
    [ 0,  -1,    1,    0,   -1],  # s6  (orange: staying costs -1)
    [ 0,   0,   -1,   -1,   -1],  # s7  (orange)
    [ 0,   1,   -1,   -1,    0],  # s8
    [-1,  -1,   -1,    0,    1],  # s9  (blue: staying pays +1)
], dtype=float)

Two small helpers to print a value function and a policy as a 3x3 grid.

In [4]:
def show_values(V, label="value function"):
    print(label + ":")
    print(np.array2string(V.reshape(3, 3), precision=2, floatmode="fixed"))

def show_policy(pi, label="policy"):
    print(label + ":")
    arrows = np.array([action_arrows[a] for a in pi]).reshape(3, 3)
    for row in arrows:
        print("  " + "  ".join(row))

## The action values

Everything below rests on one line. For a state $s$, the five action values are

$$q(s, a) = \texttt{REWARD}[s, a] + \gamma\, V\big(\texttt{NEXT}[s, a]\big), \qquad a = 0,\ldots,4,$$

a single term each because the environment is deterministic. Policy evaluation used
$q(s, \pi(s))$; value iteration uses $\max_a q(s, a)$.

In [5]:
def action_values(V, s):
    """The five q(s,a) for a given state, as a length-5 array."""
    return np.array([REWARD[s, a] + gamma * V[NEXT[s, a]] for a in range(n_actions)])

## The algorithm

Initialize $V \equiv 0$, then sweep. As in the book, states are updated **in place**: the new
$V(s)$ is immediately visible to the states that come after it in the same sweep.

In [ ]:
V = np.zeros(n_states)

sweep = 0
while True:
    delta = 0.0
    for s in range(n_states):
        v = V[s]
        V[s] = np.max(action_values(V, s))
        delta = max(delta, abs(v - V[s]))
    sweep += 1

    if sweep <= 5:
        print(f"--- sweep {sweep},  delta = {delta:.4f}")
        show_values(V)
        print()

    if delta < theta:
        break

print(f"converged after {sweep} sweeps")

v_old = np.zeros(n_states)

sweep_sync = 0
while True:
    v_new = np.array([np.max(action_values(v_old, s)) for s in range(n_states)])
    delta = np.max(np.abs(v_new - v_old))
    v_old = v_new
    sweep_sync += 1
    if delta < theta:
        break

print(f"synchronous: {sweep_sync} sweeps")
show_values(v_new)

--- sweep 1,  delta = 1.0000
value function:
[[0.00 0.00 0.00]
 [0.00 0.00 1.00]
 [0.00 1.00 1.00]]

--- sweep 2,  delta = 0.9000
value function:
[[0.00 0.00 0.00]
 [0.00 0.90 1.90]
 [0.90 1.90 1.90]]

--- sweep 3,  delta = 0.8100
value function:
[[0.00 0.81 0.73]
 [0.81 1.71 2.71]
 [1.71 2.71 2.71]]

--- sweep 4,  delta = 0.7290
value function:
[[0.73 1.54 1.44]
 [1.54 2.44 3.44]
 [2.44 3.44 3.44]]

--- sweep 5,  delta = 0.6561
value function:
[[1.39 2.20 2.10]
 [2.20 3.10 4.10]
 [3.10 4.10 4.10]]

converged after 133 sweeps


## Output the policy

The box returns a deterministic policy at the end, greedy with respect to the converged $V$:

$$\pi(s) = \arg\max_a \; \texttt{REWARD}[s, a] + \gamma\, V\big(\texttt{NEXT}[s, a]\big).$$

In [7]:
pi = np.array([np.argmax(action_values(V, s)) for s in range(n_states)])

show_values(V, "optimal value function v*")
print()
show_policy(pi, "optimal policy pi*")
print()
print("action per state:", [action_names[a] for a in pi])

optimal value function v*:
[[ 7.29  8.10  8.00]
 [ 8.10  9.00 10.00]
 [ 9.00 10.00 10.00]]

optimal policy pi*:
  >  v  v
  >  v  v
  >  >  o

action per state: ['right', 'down', 'down', 'right', 'down', 'down', 'right', 'right', 'stay']


This is exactly the $v_*$ and $\pi_*$ produced by policy iteration:

$$v_* = \begin{bmatrix} 7.29 & 8.10 & 8.00 \\ 8.10 & 9.00 & 10.00 \\ 9.00 & 10.00 & 10.00 \end{bmatrix}$$

with $s_3$ cutting down through the orange cell $s_6$. Two different algorithms, one
optimal value function — as the theory promises, since $v_*$ is unique.

## Experiment: the policy settles long before the values do

The $133$ sweeps above are spent squeezing the last decimals out of $V$. But the
*ranking* of the actions stabilises almost immediately, and the ranking is all the policy
needs. Let us record the greedy policy after every sweep and see when it stops changing.

In [9]:
V_track = np.zeros(n_states)
pi_star = pi.copy()

for k in range(1, 21):
    for s in range(n_states):
        V_track[s] = np.max(action_values(V_track, s))
    greedy = np.array([np.argmax(action_values(V_track, s)) for s in range(n_states)])
    same = "optimal" if np.array_equal(greedy, pi_star) else "not yet"
    print(f"sweep {k:2d}:  greedy policy is {same}")

sweep  1:  greedy policy is not yet
sweep  2:  greedy policy is optimal
sweep  3:  greedy policy is optimal
sweep  4:  greedy policy is optimal
sweep  5:  greedy policy is optimal
sweep  6:  greedy policy is optimal
sweep  7:  greedy policy is optimal
sweep  8:  greedy policy is optimal
sweep  9:  greedy policy is optimal
sweep 10:  greedy policy is optimal
sweep 11:  greedy policy is optimal
sweep 12:  greedy policy is optimal
sweep 13:  greedy policy is optimal
sweep 14:  greedy policy is optimal
sweep 15:  greedy policy is optimal
sweep 16:  greedy policy is optimal
sweep 17:  greedy policy is optimal
sweep 18:  greedy policy is optimal
sweep 19:  greedy policy is optimal
sweep 20:  greedy policy is optimal


The greedy policy is already optimal from the **second** sweep, while $V$ needs $133$ to reach $\theta = 10^{-6}$. This is the observation behind Sutton's Figure 4.1 and the reason truncating policy evaluation costs nothing: you can stop updating values long before they converge and still act optimally. Of course, you only know this in hindsight (the stopping rule has to watch $\Delta$, because in general a late change of ranking is possible).

Why so many sweeps? The self-loop at $s_9$ (staying on the blue cell) makes the error decay like $\gamma^k = 0.9^k$, and $0.9^{133} \approx 10^{-6}$.